# Task 1:⁠ ⁠Data Collection & Preprocessing
## Retrieve voting data from the Danish Parliament API

| Entity       | Purpose                | Key Fields                                     |
|--------------|------------------------|------------------------------------------------|
| `Afstemning` | Voting sessions        | `konklusion`, `vedtaget`, `mødeid`, `typeid`   |
| `Stemme`     | Individual votes       | `typeid`, `aktørid`, `afstemningid`            |
| `Aktør`      | Politicians/parties    | `id`, `navn`                      |
| `Sag`        | Legislative cases      | `afgørelse`, `afstemningskonklusion`, `typeid` |
| `Møde`       | Parliamentary meetings | `dato`, `nummer`, `periodeid`                  |

In [ ]:
import json
import os
import requests
import pandas as pd
from tqdm.auto import tqdm

def fetch(entity, filter, expand=None, select=None):
    path = f"raw/{entity}.json"
    url = "https://oda.ft.dk/api/" + entity
    params = {
        "$format": "json", 
        "$inlinecount": "allpages", 
        "$filter": filter,
        "$select": select,
        "$expand": expand

    }

    if os.path.exists(path):
        return json.load(open(path))

    out, bar = [], None
    while url:
        d = requests.get(url, params=params, timeout=60).json()
        bar = bar or tqdm(total=int(d["odata.count"]), desc=entity)
        out += d["value"]
        bar.update(len(d["value"]))
        url, params = d.get("odata.nextLink"), None

    os.makedirs("data", exist_ok=True)
    json.dump(out, open(path, "w"))
    return out

: 

## Obtain sessions and votes
We fetch the voting sessions (Afstemning) that were held in meetings (Møde) in the 2022-2025 legislation period. Then, we fetch the individual votes (Stemme) for those sessions.

Note that a voting session is not necessarily a law-passing session, it can be on a procedural matter.

### Time range: `2022-11-01` to `2025-10-01` (exclusive)
- **Start**: the general election of 1 November 2022. Earlier votes belong to the previous parliament, with different MPs.
- **End**: the legislature itself lasted until the election of 24 March 2026, but we stop before the 2025–26 parliamentary year (which opened in October 2025) because:
  1. We keep three **complete** parliamentary years (2022–23, 2023–24, 2024–25). The 2025–26 year was cut short when the election was called in February 2026.
  2. Votes just before an election are shaped by campaigning (symbolic motions, parties emphasising their differences), which could look like maverick behaviour.
  3. MPs who switched party in late 2025 or early 2026 can serve as an **out-of-sample check**: we can test whether our methods flagged them as mavericks from earlier votes alone.

In [ ]:
START, END = "2022-11-01T00:00:00", "2025-10-01T00:00:00"

# Fetch all the voting session in the 2022-2025 legislature
sessions = fetch("Afstemning", f"Møde/dato ge datetime'{START}' and Møde/dato lt datetime'{END}'", expand="Møde")
ids = {v["id"] for v in sessions}

# Obtain each invidivual vote from all sessions
votes = fetch("Stemme", f"afstemningid ge {min(ids)} and afstemningid le {max(ids)}")
votes = [s for s in votes if s["afstemningid"] in ids]

print(len(sessions), len(votes), len(votes) / len(sessions))

## Clean and structure the data
Merge votes with voting session information (verbose conclusion, whether the vote was passed, and the date).

In [ ]:
sessions_df = pd.DataFrame({
    "session":    [s["id"] for s in sessions],
    "conclusion": [s["konklusion"] for s in sessions],
    "passed":     [s["vedtaget"] for s in sessions],
    "date":       [s["Møde"]["dato"] for s in sessions],
})

votes_df = pd.DataFrame(votes).rename(columns={
    "afstemningid": "session", 
    "aktørid": "actor", 
    "typeid": "vote"
})[["session", "actor", "vote"]]

df = votes_df.merge(sessions_df, on="session", how="left")
df["date"] = pd.to_datetime(df["date"])
df

## Match MPs with their political parties
Each politician is an Aktør and it's linked to different entities (also Aktør), including parties. To find a MP's party, we need to find the rows that link his ID in the fraaktørid to an Aktør with type_id = 4 in the tilaktørid. 

Since a MP can change a party over time, this association has a start time and (sometimes) a end time. We use this information to determine the party association at the time of the vote. The dataset can present multiple party associations at the same time (e.g., multiple relationships with no end), so we only keep the one that started the closest to the vote date.

In [ ]:
# Party memberships overlapping 2022–25 (MP -> party, with dates)
rel = pd.DataFrame(fetch(
    "AktørAktør",
    f"rolleid eq 15 and TilAktør/typeid eq 4 "
    f"and startdato lt datetime'{END}' "
    f"and (slutdato ge datetime'{START}' or slutdato eq null)",
    expand="TilAktør",
    select="fraaktørid,startdato,slutdato,TilAktør/navn",
))

rel = pd.DataFrame({
    "actor":     rel["fraaktørid"],
    "party":     rel["TilAktør"].str["navn"],
    "startdato": pd.to_datetime(rel["startdato"]),
    "slutdato":  pd.to_datetime(rel["slutdato"]).fillna(pd.Timestamp.max),  # no end date = still open
})

# Find the active membership (most recent one if several) on the date of the vote
m = df[["session", "actor", "date"]].merge(rel, on="actor")
m = m[(m["date"] >= m["startdato"]) & (m["date"] <= m["slutdato"])]
m = m.sort_values("startdato").drop_duplicates(["session", "actor"], keep="last")
m = m[["session", "actor", "party"]]

# Merge the party to every vote
df = df.drop(columns="party", errors="ignore")  # prevent re-run errors
df = df.merge(m, on=["session", "actor"], how="left")

df

## Construct the MP × voting matrix
The matrix contains 233.042 rows and 7 columns which represent:

| Column | Type | Description |
|---|---|---|
| `session` | int | Voting session id (`Afstemning.id`). Several sessions can happen in the same meeting. |
| `actor` | int | MP id (`Aktør.id`, type 5 = person). |
| `vote` | int | How the MP voted (`Stemmetype`): 1 = For, 2 = Imod (against), 3 = Fravær (absent), 4 = Hverken for eller imod (abstain). |
| `conclusion` | str | Official outcome text of the session, incl. vote counts per party (`Afstemning.konklusion`). |
| `passed` | bool | Whether the proposal was adopted (`Afstemning.vedtaget`). |
| `date` | datetime | Date and time of the meeting the session belongs to (`Møde.dato`). |
| `party` | str | The MP's party **on the date of the vote**, from `AktørAktør` memberships (most recent one if several overlap). NaN if no membership covers that date. |


In [ ]:
df.to_csv("votes.csv", index=False)